In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.cross_decomposition import PLSRegression

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

import os

plt.style.use("default")

In [ ]:
!mkdir -p data
!mkdir -p outputs

In [ ]:
import pandas as pd
import os

data_path = "/content/data"

files = {
    "ADANIPOWER": "ADANIPOWER-upl.xlsx",
    "ASIANPAINT": "ASIANPAINT-upl.xlsx",
    "LT": "LT-upl.xlsx",
    "TCS": "TCS-upl.xlsx",
    "WIPRO": "WIPRO-upl.xlsx",
    "NIFTY": "NIFTY-50-upl.xlsx"
}

dfs = {}

for name, file in files.items():
    df = pd.read_excel(os.path.join(data_path, file))

    # 1. Strip whitespace from column names
    df.columns = df.columns.str.strip()

    # 2. Drop rows where Date is not a valid date (e.g. "ADANIPOWER")
    df = df[df["Date"].astype(str).str.match(r"\d{2}-\w{3}-\d{4}")]

    # 3. Convert Date properly
    df["Date"] = pd.to_datetime(df["Date"], format="%d-%b-%Y")

    # 4. Sort & clean
    df = df.sort_values("Date").drop_duplicates().reset_index(drop=True)

    # 5. Keep only required columns and standardize names
    df = df.rename(columns={
        "Close Price": "Close",
        "Open Price": "Open",
        "High Price": "High",
        "Low Price": "Low"
    })

    dfs[name] = df

    print(f"\n{name} loaded successfully")
    print(df[["Date", "Open", "High", "Low", "Close"]].head(2))



ADANIPOWER loaded successfully
Empty DataFrame
Columns: [Date, Open, High, Low, Close]
Index: []

ASIANPAINT loaded successfully
Empty DataFrame
Columns: [Date, Open, High, Low, Close]
Index: []

LT loaded successfully
Empty DataFrame
Columns: [Date, Open, High, Low, Close]
Index: []

TCS loaded successfully
Empty DataFrame
Columns: [Date, Open, High, Low, Close]
Index: []

WIPRO loaded successfully
Empty DataFrame
Columns: [Date, Open, High, Low, Close]
Index: []

NIFTY loaded successfully
Empty DataFrame
Columns: [Date, Open, High, Low, Close]
Index: []


In [ ]:
merged = dfs["ADANIPOWER"][["Date", "Close"]].rename(columns={"Close": "ADANIPOWER"})

for stock in ["ASIANPAINT", "LT", "TCS", "WIPRO", "NIFTY"]:
    merged = merged.merge(
        dfs[stock][["Date", "Close"]].rename(columns={"Close": stock}),
        on="Date",
        how="outer"
    )

merged = merged.sort_values("Date").reset_index(drop=True)
merged.head()


,Date,ADANIPOWER,ASIANPAINT,LT,TCS,WIPRO,NIFTY


In [ ]:
for name, df in dfs.items():
    print("\n========================")
    print(name)
    print("Shape:", df.shape)
    print(df.head(3))
    print(df["Date"].head(3))
    print("Date dtype:", df["Date"].dtype)



ADANIPOWER
Shape: (0, 13)
Empty DataFrame
Columns: [Symbol, Series, Date, Prev Close, Open, High, Low, Last Price, Close, Average Price, Total Traded Quantity, Turnover ₹, No. of Trades]
Index: []
Series([], Name: Date, dtype: datetime64[ns])
Date dtype: datetime64[ns]

ASIANPAINT
Shape: (0, 13)
Empty DataFrame
Columns: [Symbol, Series, Date, Prev Close, Open, High, Low, Last Price, Close, Average Price, Total Traded Quantity, Turnover ₹, No. of Trades]
Index: []
Series([], Name: Date, dtype: datetime64[ns])
Date dtype: datetime64[ns]

LT
Shape: (0, 13)
Empty DataFrame
Columns: [Symbol, Series, Date, Prev Close, Open, High, Low, Last Price, Close, Average Price, Total Traded Quantity, Turnover ₹, No. of Trades]
Index: []
Series([], Name: Date, dtype: datetime64[ns])
Date dtype: datetime64[ns]

TCS
Shape: (0, 13)
Empty DataFrame
Columns: [Symbol, Series, Date, Prev Close, Open, High, Low, Last Price, Close, Average Price, Total Traded Quantity, Turnover ₹, No. of Trades]
Index: []
Seri

In [ ]:
print("Merged shape:", merged.shape)
print(merged.head())


Merged shape: (0, 7)
Empty DataFrame
Columns: [Date, ADANIPOWER, ASIANPAINT, LT, TCS, WIPRO, NIFTY]
Index: []


In [ ]:
print("Before NaN handling:", merged.shape)

# Option 1: forward fill (market realistic)
merged_ffill = merged.fillna(method="ffill")

# Option 2: drop rows with any NaN
merged_clean = merged_ffill.dropna()

print("After NaN handling:", merged_clean.shape)


Before NaN handling: (0, 7)
After NaN handling: (0, 7)


/tmp/ipython-input-1786436988.py:4: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged_ffill = merged.fillna(method="ffill")


In [ ]:
merged = merged_clean.copy()
merged.set_index("Date", inplace=True)


In [ ]:
print(merged.shape)
print(merged.head())


(0, 6)
Empty DataFrame
Columns: [ADANIPOWER, ASIANPAINT, LT, TCS, WIPRO, NIFTY]
Index: []


In [21]:
corr = merged.drop(columns=["NIFTY"]).corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("Stock Price Correlation Heatmap")
plt.tight_layout()
plt.savefig("../outputs/correlation_heatmap.png")
plt.show()

In [ ]:
df_lag = merged.copy()

for col in df_lag.columns:
    df_lag[col + "_lag1"] = df_lag[col].shift(1)

df_lag.dropna(inplace=True)
df_lag.head()


In [ ]:
X = df_lag[["WIPRO_lag1"]]
y = df_lag["ASIANPAINT"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=True, random_state=42
)

models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.01)
}

results_r1 = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    results_r1.append({
        "Model": name,
        "R2": r2_score(y_test, preds),
        "RMSE": mean_squared_error(y_test, preds, squared=False),
        "MAE": mean_absolute_error(y_test, preds)
    })

pd.DataFrame(results_r1)


In [ ]:
wipro_ohlc = dfs["WIPRO"][["Date", "Open", "High", "Low", "Close"]].copy()
wipro_ohlc["Date"] = pd.to_datetime(wipro_ohlc["Date"])
wipro_ohlc = wipro_ohlc.sort_values("Date").drop_duplicates()

for col in ["Open", "High", "Low", "Close"]:
    wipro_ohlc[col + "_lag1"] = wipro_ohlc[col].shift(1)

wipro_ohlc.dropna(inplace=True)

df_r2 = merged.merge(wipro_ohlc[["Date", "Open_lag1", "High_lag1", "Low_lag1", "Close_lag1"]],
                     left_index=True, right_on="Date", how="inner").set_index("Date")

X = df_r2[["Open_lag1", "High_lag1", "Low_lag1", "Close_lag1"]]
y = df_r2["ASIANPAINT"]


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=True, random_state=42
)

results_r2 = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    results_r2.append({
        "Model": name,
        "R2": r2_score(y_test, preds),
        "RMSE": mean_squared_error(y_test, preds, squared=False),
        "MAE": mean_absolute_error(y_test, preds)
    })

pd.DataFrame(results_r2)


In [ ]:
X = df_lag[[c for c in df_lag.columns if "lag1" in c and "NIFTY" not in c]]
y = df_lag["NIFTY"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=True, random_state=42
)

lr = LinearRegression()
lr.fit(X_train, y_train)
preds = lr.predict(X_test)

print("R2:", r2_score(y_test, preds))
print("RMSE:", mean_squared_error(y_test, preds, squared=False))
print("MAE:", mean_absolute_error(y_test, preds))


In [ ]:
X_const = sm.add_constant(X)

vif = pd.DataFrame()
vif["Feature"] = X_const.columns
vif["VIF"] = [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]
vif
